# 02 — Live deployment, text mode

Live camera inference on the Kria KV260 with text-based status reporting.
This is the **maximum-FPS** variant — no per-frame rendering, no JPEG encode,
no widget updates. End-to-end throughput is bounded by either the camera
(60 fps for the Brio at 480p MJPG) or the DPU (~12 ms total at imgsz=320, so
~80 fps ceiling).

For the **interactive video** variant (slower, but with bounding boxes
overlaid live and parameter sliders), see `03_deploy_visual.ipynb`.

**Architecture**: heavy lifting (`Preprocessor`, `decode_yolov5u`,
`ThreadedCamera`, `ModelRunner`) is factored into the `lpr_pipeline.deploy`
package. This notebook orchestrates them and renders an HTML status block
that refreshes once per second.

**Prerequisites** — handled by Pass 5's Kria scripts:

| Task | Where |
|---|---|
| VAI 3.5 runtime + DPU-PYNQ | `scripts/kria/01_install_vai35.sh` |
| USB autosuspend, CPU governor, camera tuning | `scripts/kria/02_apply_tuning.sh` |
| Persistence across reboots | `scripts/kria/03_install_systemd.sh` |
| xmodel synced from laptop | `scripts/host/03_sync_to_kria.sh` |
| Jupyter launched as root + xmutil unload + env sourcing | `scripts/kria/run_live.sh <variant>` |

**Stop the live loop**: stop button (■) in the JupyterLab toolbar, or
*Kernel → Interrupt*. Final stats are printed.


## 1. Configuration

Edit `VARIANT` to pick the model, then run the rest top-to-bottom.

If launched via `scripts/kria/run_live.sh <variant>`, the script sets
`LPR_VARIANT` and `LPR_XMODEL` and these values are picked up from there
automatically.

`CONF_THRESH`, `IOU_THRESH`, and `MAX_DETECTIONS` are *initial* values; in
the visual notebook (`03_deploy_visual.ipynb`) you can adjust them live via
sliders without restarting the cell.


In [ ]:
import os

VARIANT        = os.environ.get("LPR_VARIANT", "yolov5n")
CONF_THRESH    = 0.30
IOU_THRESH     = 0.45
MAX_DETECTIONS = 300

# Status display refresh rates
STATUS_DT      = 1.00     # seconds between HTML status refreshes
DET_LOG_DT     = 0.40     # min seconds between detection event prints

print(f"VARIANT        = {VARIANT}")
print(f"CONF_THRESH    = {CONF_THRESH}")
print(f"IOU_THRESH     = {IOU_THRESH}")
print(f"MAX_DETECTIONS = {MAX_DETECTIONS}")


## 2. Suppress glog noise

The Vitis AI runtime prints `Check failed: r == 0 cannot set read range!` on
every model load. Harmless ("fingerprint-verification quirk") but floods the
notebook output. Setting these env vars before any `vart`/`xir` import keeps
only FATAL-level glog messages.


In [ ]:
import os
os.environ['GLOG_minloglevel']     = '3'
os.environ['GLOG_logtostderr']     = '0'
os.environ['GLOG_stderrthreshold'] = '3'


## 3. Imports + repo location

The repo root is normally `/home/ubuntu/KriaKv260_Model_Compiler` on the Kria.
We add it to `sys.path` so `lpr_pipeline.deploy.*` is importable.


In [ ]:
import sys
from pathlib import Path

candidates = [
    os.environ.get("REPO_ROOT"),
    "/home/ubuntu/KriaKv260_Model_Compiler",
    str(Path.cwd().parent),
    str(Path.cwd()),
]
for cand in candidates:
    if cand and (Path(cand) / "lpr_pipeline").is_dir():
        REPO_ROOT = cand
        if cand not in sys.path:
            sys.path.insert(0, cand)
        break
else:
    raise RuntimeError(
        "Could not locate lpr_pipeline. Set REPO_ROOT env var or run via "
        "scripts/kria/run_live.sh which sets it for you."
    )

print(f"REPO_ROOT = {REPO_ROOT}")

from lpr_pipeline.shared.models import get_spec

spec = get_spec(VARIANT)
XMODEL = os.environ.get(
    "LPR_XMODEL",
    f"/home/ubuntu/xmodels_vai35/{VARIANT}/{VARIANT}_kv260.xmodel",
)

print(f"spec   = family={spec.family} imgsz={spec.imgsz} "
      f"nc={spec.nc} reg_max={spec.reg_max}")
print(f"xmodel = {XMODEL}")

if not Path(XMODEL).exists():
    raise FileNotFoundError(
        f"xmodel missing: {XMODEL}\n"
        f"From your laptop, sync it:\n"
        f"  bash scripts/host/03_sync_to_kria.sh ubuntu@<kria-ip> {VARIANT}"
    )

import time, threading
import numpy as np
import cv2
from IPython.display import display, HTML
from pynq_dpu import DpuOverlay

from lpr_pipeline.deploy import (
    ModelRunner, ThreadedCamera,
    draw_detections, draw_stats_overlay,
)

print(f"OpenCV {cv2.__version__}")


## 4. Load DPU overlay (program the FPGA)

The DPU bitstream is loaded once. After this cell runs, the FPGA fabric is
configured and we can hot-swap xmodels onto it via `overlay.load_model()`
without reprogramming the hardware.


In [ ]:
overlay = DpuOverlay("dpu.bit")
print("DPU overlay loaded — FPGA programmed and ready for xmodels.")


## 5. Build the ModelRunner and warm it up

`ModelRunner` ties together the preprocessor (letterbox + normalize), the
DPU runner, and the decoder (`decode_yolov5u`). Warmup runs 5 inferences on
random input to settle JIT + caches.


In [ ]:
runner = ModelRunner(spec, XMODEL, overlay)

print(f"\nModelRunner built:")
print(f"  input  dims = {runner.input_dims}")
for i, d in enumerate(runner.output_dims):
    print(f"  output[{i}] = {d}")

print(f"\nWarming up (5 runs):")
warmup_times = runner.warmup(n=5, print_each=True)
print(f"\nSteady-state ≈ {min(warmup_times[2:]):.2f} ms (best of last 3)")


## 6. Pure inference benchmark (DPU + decode ceiling)

Measures maximum inference rate fed an in-memory frame — no camera, no
display. The reportable number is `mean_ms`; throughput is `1000 / mean_ms`.


In [ ]:
def benchmark_pure(n=200, warmup=20):
    # Pure-inference benchmark with per-stage breakdown.
    frame = np.random.randint(0, 256, (480, 640, 3), dtype=np.uint8)
    for _ in range(warmup):
        runner.infer(frame)

    total_times = np.empty(n, dtype=np.float64)
    pre_times   = np.empty(n, dtype=np.float64)
    dpu_times   = np.empty(n, dtype=np.float64)
    dec_times   = np.empty(n, dtype=np.float64)

    for i in range(n):
        t0 = time.perf_counter()
        _, t = runner.infer(frame)
        total_times[i] = (time.perf_counter() - t0) * 1000
        pre_times[i]   = t["preprocess"]
        dpu_times[i]   = t["dpu"]
        dec_times[i]   = t["decode"]

    def stats(arr, label):
        print(f"  {label:>10s}  "
              f"mean={arr.mean():6.2f}  "
              f"p50={np.percentile(arr,50):6.2f}  "
              f"p95={np.percentile(arr,95):6.2f}  "
              f"p99={np.percentile(arr,99):6.2f}")

    print(f"=== {VARIANT} pure inference (n={n}) — timings in ms ===")
    stats(total_times, "total")
    stats(pre_times,   "preprocess")
    stats(dpu_times,   "dpu")
    stats(dec_times,   "decode")
    print(f"\n  → throughput = {1000 / total_times.mean():6.1f} fps  "
          f"(p95: {1000 / np.percentile(total_times, 95):.1f} fps)")
    return total_times, pre_times, dpu_times, dec_times

bench_total, bench_pre, bench_dpu, bench_dec = benchmark_pure()


## 7. Live loop — text status only (max throughput)

Threaded camera (BUFFERSIZE=4, MJPG, 60 fps) feeds the runner; the HTML
status block refreshes once per second; detection events stream below
(throttled to once per 0.4 s).

This loop is the **thesis benchmark variant** — no per-frame rendering. End-
to-end FPS should equal camera FPS (~60) on yolov5n.

**Stop**: stop button (■) in the JupyterLab toolbar.


In [ ]:
# Release any prior camera handle from a failed earlier run
try:
    cam.close()
    print("(closed prior camera)")
except (NameError, AttributeError):
    pass

cam = ThreadedCamera()
print(f"  camera: {cam.actual_width}x{cam.actual_height} "
      f"@ {cam.actual_fps:.0f} fps  BUFFERSIZE=4")

status = display(
    HTML("<pre style='font-family:monospace'>starting...</pre>"),
    display_id=True,
)

n_inf, n_dets_total, n_frames_with_d = 0, 0, 0
unique_ids = set()
ema_pre, ema_dpu, ema_dec = 0.0, 0.0, 0.0

t_start         = time.perf_counter()
last_status_t   = t_start
last_dets_log_t = 0.0

print(f"\n[ live {VARIANT}  •  press stop (■) to end ]\n")

try:
    while True:
        frame, fid = cam.read_new()
        if frame is None:
            time.sleep(0.001)
            continue
        unique_ids.add(fid)

        dets, t = runner.infer(frame, conf=CONF_THRESH, iou=IOU_THRESH,
                                max_detections=MAX_DETECTIONS)
        n_inf += 1

        if ema_pre:
            ema_pre = 0.9 * ema_pre + 0.1 * t["preprocess"]
            ema_dpu = 0.9 * ema_dpu + 0.1 * t["dpu"]
            ema_dec = 0.9 * ema_dec + 0.1 * t["decode"]
        else:
            ema_pre, ema_dpu, ema_dec = t["preprocess"], t["dpu"], t["decode"]

        if dets:
            n_dets_total    += len(dets)
            n_frames_with_d += 1

        now = time.perf_counter()

        if dets and (now - last_dets_log_t) >= DET_LOG_DT:
            last_dets_log_t = now
            for x1, y1, x2, y2, conf, cls_idx in dets:
                cls_name = "plate" if spec.nc == 1 else str(cls_idx)
                print(f"  ▶ {cls_name}  "
                      f"({int(x1):3d},{int(y1):3d}) → ({int(x2):3d},{int(y2):3d})  "
                      f"conf={conf:.3f}")

        if (now - last_status_t) >= STATUS_DT:
            last_status_t = now
            elapsed = now - t_start
            inf_fps = n_inf / elapsed
            cam_fps = len(unique_ids) / elapsed
            hit_pct = n_frames_with_d / n_inf * 100 if n_inf else 0
            ema_total = ema_pre + ema_dpu + ema_dec
            html = (
                "<pre style='font-family:monospace;font-size:13px;"
                "background:#1e1e1e;color:#d4d4d4;padding:8px;"
                "border-radius:4px'>"
                f"<b style='color:#4ec9b0'>{VARIANT}</b>  "
                f"elapsed={elapsed:6.1f}s  frames={n_inf:6d}\n"
                f"<b style='color:#dcdcaa'>inf_fps</b>={inf_fps:5.1f}  "
                f"<b style='color:#dcdcaa'>cam_fps</b>={cam_fps:5.1f}  "
                f"<b style='color:#dcdcaa'>theoretical_max</b>="
                f"{1000/ema_total if ema_total else 0:5.1f} fps\n"
                f"<b style='color:#9cdcfe'>pre</b>={ema_pre:4.2f}  "
                f"<b style='color:#9cdcfe'>dpu</b>={ema_dpu:5.2f}  "
                f"<b style='color:#9cdcfe'>dec</b>={ema_dec:4.2f}  "
                f"(<b style='color:#9cdcfe'>total</b>={ema_total:5.2f} ms)\n"
                f"<b style='color:#c586c0'>detections</b>={n_dets_total}  "
                f"<b style='color:#c586c0'>hit_rate</b>={hit_pct:5.1f}%"
                "</pre>"
            )
            status.update(HTML(html))

except KeyboardInterrupt:
    print("\n[ stopped by user ]")
finally:
    cam.close()
    elapsed = time.perf_counter() - t_start
    inf_fps = n_inf / elapsed if elapsed > 0 else 0
    cam_fps = len(unique_ids) / elapsed if elapsed > 0 else 0
    hit_pct = n_frames_with_d / n_inf * 100 if n_inf else 0
    ema_total = ema_pre + ema_dpu + ema_dec

    print(f"\n=== final stats — {VARIANT} ===")
    print(f"  total time            : {elapsed:8.2f} s")
    print(f"  frames inferred       : {n_inf:8d}")
    print(f"  unique camera frames  : {len(unique_ids):8d}")
    print(f"  inference fps         : {inf_fps:8.2f}")
    print(f"  camera fps            : {cam_fps:8.2f}")
    print(f"  avg preprocess (ms)   : {ema_pre:8.2f}")
    print(f"  avg dpu        (ms)   : {ema_dpu:8.2f}")
    print(f"  avg decode     (ms)   : {ema_dec:8.2f}")
    print(f"  avg total      (ms)   : {ema_total:8.2f}")
    print(f"  theoretical max       : {1000/ema_total if ema_total else 0:8.2f} fps")
    print(f"  total detections      : {n_dets_total:8d}")
    print(f"  frames with object    : {n_frames_with_d:8d}")
    print(f"  hit rate              : {hit_pct:8.2f} %")


## 8. Notes

### Switching models

Change `VARIANT` in cell 1 and **restart the kernel**, then run all cells
again. The DPU configuration is per-xmodel and the runner state needs a
clean slate.

When launching via `scripts/kria/run_live.sh <variant>`, the env var
`LPR_VARIANT` is set automatically.

### Reading the numbers

| Number | What it means | Bound by |
|---|---|---|
| `inf_fps` | End-to-end throughput | camera or DPU, whichever is slower |
| `cam_fps` | Camera's unique-frame rate | Brio @ 60 |
| `dpu` | DPU compute time per inference | model size + DPU config |
| `pre` + `dec` | CPU work per inference | the four Cortex-A53s |
| `theoretical_max` | Pipeline ceiling | `1000 / (pre + dpu + dec)` |

If `inf_fps ≈ cam_fps ≈ 60` on yolov5n → pipeline is camera-bound; the
DPU has spare capacity.

### Where stuff lives

| Thing | File |
|---|---|
| `Preprocessor`, `unletterbox` | `lpr_pipeline/deploy/preprocess.py` |
| `decode_yolov5u` | `lpr_pipeline/deploy/decoders.py` |
| `ThreadedCamera` | `lpr_pipeline/deploy/camera.py` |
| `ModelRunner` | `lpr_pipeline/deploy/runner.py` |
| `draw_detections`, `draw_stats_overlay` | `lpr_pipeline/deploy/draw.py` |
| Model specs | `lpr_pipeline/shared/models.py` |

### What this notebook deliberately *doesn't* do

- **System tuning**: handled by `scripts/kria/02_apply_tuning.sh`, persisted
  by the `kriakv260-tuning.service` systemd unit.
- **Per-frame video rendering**: that's the `03_deploy_visual.ipynb` job.
  This notebook is the throughput benchmark.
- **Evaluation / mAP**: deferred to a future pass.
- **YOLOX**: spec exists, decoder doesn't yet. yolov5n and yolov5s only.
